**Dùng Mistral để cải thiện câu trả lời trong SFT Data**

In [1]:
NUM_SAMPLES = 1000

import json
import datasets
from datasets import Dataset
import random

data_list = []
with open("/kaggle/input/nignigavlcsyntheticdata/Synthetic Data Train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        data_list.append(data)

sample_data = random.sample(data_list, NUM_SAMPLES)
dataset = Dataset.from_list(sample_data) 


In [2]:
!pip install mistralai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 1.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.0/461.0 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.37.0
    Uninstalling opentelemetry-proto-1.37.0:
      Successfully uninstalled opentelemetry-proto-1.37.0
  Attempting uninstall: opentelemetry-exporter-otlp-proto-common
    Found existi

In [ ]:
from mistralai import Mistral
import os

def quick_reply(prompt, model="mistral-large-latest", api_key=None):
    client = Mistral(api_key=os.environ.get('MISTRAL_API_KEY'))
    inputs = [{"role": "user", "content": prompt}]
    completion_args = {
        "temperature": 0.7,
        "max_tokens": 2048,
        "top_p": 1
    }
    response = client.beta.conversations.start(
        inputs=inputs,
        model=model,
        instructions="",
        completion_args=completion_args,
        tools=[],
    )

    return response.outputs[0].content if response.outputs else None

if __name__ == "__main__":
    print(quick_reply("Xin chào!"))


Xin chào! 😊 Tôi là Mistral, một mô hình ngôn ngữ lớn của Mistral AI. Bạn cần giúp gì hôm nay?


In [4]:
def improve_ans(question, answer):
    prompt = f"""
    Bạn là một trợ lý AI chuyên nghiệp, thân thiện và hữu ích.
    Nhiệm vụ: Viết lại đoạn văn bản dưới đây sao cho:
    1. GIỮ NGUYÊN ý nghĩa và thông tin kỹ thuật (Tuyệt đối không bịa đặt sai lệch).
    2. Thay đổi giọng văn: Trở nên nhiệt tình, quan tâm người dùng hơn (thay vì cộc lốc).
    3. Cải thiện trình bày: Trình bày đơn giản, không cầu kì (Tuyệt đối không in đậm (markdown), không dùng icon, không dùng kí tự đặc biệt như '*', ...).
    4. Trả lời với văn phong giống một bác sĩ. Hãy nhớ ngắn gọn và xúc tích, phù hợp với câu hỏi.
    Chỉ trả lại đoạn văn đã sửa.
    Câu hỏi: {question}
    Đoạn văn cần được sửa: {answer}
    """

    return quick_reply(prompt)

In [5]:
from tqdm import tqdm

dpo_data = []
for data in tqdm(dataset):
    question = data["question"]
    rejected = data["answer"]
    chosen = improve_ans(question, rejected)
    print(question,"\n\n", rejected,"\n\n", chosen,"\n\n")
    dpo_data.append({
        "question": question,
        "chosen": chosen,
        "rejected": rejected
    })

dpo_dataset = Dataset.from_list(dpo_data)
dpo_dataset.to_json("dpo.jsonl",lines=True)

  0%|          | 1/1000 [00:10<2:56:37, 10.61s/it]

Phân loại phù bạch mạch và nêu nguyên nhân gây ra từng loại? 

 Phù bạch mạch được phân loại gồm dạng nguyên phát và dạng thứ phát.
4.1. Phù bạch mạch nguyên phát
Do một khiếm khuyết tại bản thân mạch bạch huyết hoặc hạch bạch huyết. Phù bạch mạch di truyền và các tình trạng liên quan di truyền.
+ Thể dị bội nhiễm sắc thể: hội chứng Klinefelter, hội chứng Turner, Trisomy 21 (hội chứng Down), Trisomy 13 (hội chứng Patau), Trisomy 18 (hội chứng Edwards).
+ Rối loạn hình thái: bệnh Milroy, hội chứng Klippel-Trénaunay, hội chứng Noonan, u xơ thần kinh type 1, hội chứng phù bạch huyết-suy giáp, bệnh Meige.
4.2. Phù bạch mạch thứ phát
Phát sinh do tắc nghẽn hoặc gián đoạn của toàn bộ hệ thống bạch huyết (kể cả ở hạch bạch huyết và cả mạch bạch huyết).
Hạch bạch huyết: Do can thiệp của thầy thuốc: xơ hóa đường bạch huyết do phẫu thuật hoặc xạ trị
+ Phù bạch mạch cánh tay: sau khi bóc tách hạch bạch huyết ở nách sau phẫu thuật cắt bỏ vú
+ Phù bạch mạch vùng chậu: sau khi bóc tách hạch vùng chậ

  0%|          | 2/1000 [00:16<2:12:43,  7.98s/it]

Làm sao để cải thiện lượng cholesterol HDL trong cơ thể? 

 Tập thể dục đều đặn có thể cải thiện cholesterol thấp và cholesterol cao. Ngoài ra, hoạt động thể chất vừa phải cũng giúp tăng cholesterol lipoprotein mật độ cao (cholesterol HDL). 

 Để cải thiện lượng cholesterol HDL (loại cholesterol tốt) trong cơ thể, tôi khuyên bạn nên bắt đầu với việc tập thể dục đều đặn. Hoạt động thể chất không chỉ giúp cân bằng cholesterol toàn phần mà còn đặc biệt hiệu quả trong việc tăng HDL. Bạn có thể thử các bài tập vừa phải như đi bộ nhanh, đạp xe hoặc bơi lội – những hoạt động này rất tốt cho tim mạch và dễ thực hiện hàng ngày. Hãy cố gắng duy trì ít nhất 30 phút mỗi ngày, 5 lần mỗi tuần để thấy sự thay đổi tích cực nhé! Nếu bạn có bất kỳ vấn đề sức khỏe nào khác, hãy trao đổi thêm với tôi để có lời khuyên phù hợp hơn. 




  0%|          | 3/1000 [00:30<2:53:21, 10.43s/it]

...

Teo não ở người già có thể được điều trị khỏi hoàn toàn không? 

 Bệnh teo não ở người già cũng giống như tình trạng thoái hóa ở các cơ quan khác trong cơ thể, nên rất khó có thể chữa khỏi. Khi điều trị chủ yếu là các biện pháp điều trị bằng thuốc, bổ sung vitamin, các biện pháp tâm lý trị liệu, chế độ ăn uống và sinh hoạt để cải thiện triệu chứng và giảm biến chứng làm chậm quá trình tiến triển của bệnh. 

 Teo não ở người cao tuổi là một phần tự nhiên của quá trình lão hóa, giống như nhiều cơ quan khác trong cơ thể. Thật tiếc là hiện nay y học vẫn chưa thể điều trị dứt điểm tình trạng này. Tuy nhiên, chúng ta hoàn toàn có thể làm chậm tiến triển của bệnh và cải thiện chất lượng cuộc sống cho người bệnh.

Điều trị thường tập trung vào việc kiểm soát triệu chứng và ngăn ngừa biến chứng. Bác sĩ có thể kê đơn một số loại thuốc phù hợp, kết hợp với việc bổ sung vitamin cần thiết. Ngoài ra, liệu pháp tâm lý, chế độ dinh dưỡng khoa học và thói quen sinh hoạt lành mạnh cũng đóng vai trò quan

100%|██████████| 1000/1000 [2:39:17<00:00,  9.56s/it]

Những triệu chứng nào có thể cảnh báo về sự xuất hiện của bệnh teo não? 

 Mất trí nhớĐây là triệu chứng đầu tiên và hay gặp nhất trong teo não ở người già. Người bệnh hay quên những sự việc vừa mới xảy ra. Bệnh nhân thường quên ngày tháng, các mối quan hệ, tên người thân trong nhà. Thậm chí, người bệnh còn quên cách thực hiện các hoạt động sống thường ngày như ăn uống, vệ sinh cá nhân,...Rối loạn nhận thứcNgười bệnh sẽ mất dần khả năng định hướng không gian và thời gian. Không xác định được mình đang ở đâu và đang làm gì. Lâu dần, bệnh nhân cũng không thể thực hiện được những phép tính và cách sử dụng đồ vật đơn giản.Trầm cảmTriệu chứng này thường xuất hiện ở giai đoạn sớm nhưng không ổn định. Bệnh nhân luôn có cảm giác sợ hãi, lo âu, tinh thần không ổn định. Trầm cảm do bệnh teo não ở người già cũng khiến họ sinh ra ảo giác hoặc hoang tưởng.Rối loạn ngôn ngữĐây cũng là biểu hiện sớm của chứng teo não tuổi già. Bệnh nhân gặp khó khăn khi tìm từ ngữ diễn đạt mong muốn, suy nghĩ của mìn

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

3684028